In [1]:
# # import matplotlib.pyplot as plt
# from langchain.prompts.prompt import PromptTemplate
# import copy, cloudpickle
# import sys
# import os
# current_dir = os.getcwd()
# parent_dir = os.path.join(current_dir,'..')
# sys.path.insert(0, parent_dir)
# import boicl
# from dotenv import load_dotenv
# load_dotenv()

from langchain.prompts.prompt import PromptTemplate
import copy, cloudpickle
import sys
import os

# Load env FIRST before boicl tries to instantiate OpenAI client
from dotenv import load_dotenv
load_dotenv()

# Force local boicl over any PyPI install
sys.path.insert(0, "./repos/boicl_crystal_phase_isolation")

import boicl
print("boicl loaded from:", boicl.__file__)

/Users/shane/opt/anaconda3/envs/bo_icl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


boicl loaded from: /Users/shane/repos/boicl_crystal_phase_isolation/boicl/__init__.py


# Plotting formatt

In [2]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as font_manager
import urllib.request

urllib.request.urlretrieve(
    "https://github.com/google/fonts/raw/main/ofl/ibmplexmono/IBMPlexMono-Regular.ttf",
    "IBMPlexMono-Regular.ttf",
)
fe = font_manager.FontEntry(fname="IBMPlexMono-Regular.ttf", name="plexmono")
font_manager.fontManager.ttflist.append(fe)
plt.rcParams.update(
    {
        "axes.facecolor": "#f5f4e9",
        "grid.color": "#AAAAAA",
        "axes.edgecolor": "#333333",
        "figure.facecolor": "#FFFFFF",
        "axes.grid": False,
        "axes.prop_cycle": plt.cycler("color", plt.cm.Dark2.colors),
        "font.family": fe.name,
        "figure.figsize": (3.5, 3.5 / 1.2),
        "ytick.left": True,
        "xtick.bottom": True,
    }
)


# Trajector 1: EI

## Dataset prep : find completion indices for tell, remove completed prompts from unlabled pool

In [ ]:
import pandas as pd
import numpy as np
import time

start_time = time.time()
RANDOM_SEED = 616
np.random.seed(RANDOM_SEED)

# Constants for column names
PROMPT_COL     = "Procedure"
COMPLETION_COL = "MoC(Fm3m) weight fraction"

# ── Seed results for this trajectory (human-guided campaign) ──────────────────

seed_prompt_labels = [
    # --- M7  (procedure: 800 C, 15 C/min, 0.5 hr, N2, 30 sccm, 1:1)
    ("1 g of ammonium heptamolybdate tetrahydrate dissolved in 5.5 mL of DI water. Separately, 1 g of sucrose dissolved in 2 mL of DI water at 50 °C. The two solutions were then combined. The combined solution was dried at 120 °C in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved precursor was carburized at a ramp rate of 15 °C/min to 800 °C for 0.5 hr under 30 sccm of N2 gas. After carburization the sample was cooled under 30 sccm N2 and passivated under 30 sccm 1% O2/N2 for 2 hours.",
     2.2, 1.5,             # Mo   wf, sigma
     25.7, 1.5,            # Mo2C wf, sigma
     72.1, 1.5,            # MoC  wf, sigma
     0.0, 0.0,             # MoO2 wf, sigma  (none reported)
     4.83, 0.517, 0.267,   # Rwp, GOF, chi2 (chi2 est. as GOF**2, not directly reported)
     "MoC.66"),

    # --- M12  (procedure: 600 C, 15 C/min, 2 hr, H2, 30 sccm, 1:1)
    ("1 g of ammonium heptamolybdate tetrahydrate dissolved in 5.5 mL of DI water. Separately, 1 g of sucrose dissolved in 2 mL of DI water at 50 °C. The two solutions were then combined. The combined solution was dried at 120 °C in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved precursor was carburized at a ramp rate of 15 °C/min to 600 °C for 2 hr under 30 sccm of H2 gas. After carburization the sample was cooled under 30 sccm N2 and passivated under 30 sccm 1% O2/N2 for 2 hours.",
     0.0, 0.0,             # Mo   wf, sigma  (none reported)
     5.7, 1.2,             # Mo2C wf, sigma  (none reported)
     83.8, 5.67,           # MoC  wf, sigma
     10.8, 1.39,           # MoO2 wf, sigma
     4.16, 0.548, 0.300,   # Rwp, GOF, chi2 (chi2 est. as GOF**2, not directly reported)
     "MoC.66"),

    # --- M13  (procedure: 900 C, 15 C/min, 2 hr, H2, 30 sccm, 1:1)
    ("1 g of ammonium heptamolybdate tetrahydrate dissolved in 5.5 mL of DI water. Separately, 1 g of sucrose dissolved in 2 mL of DI water at 50 °C. The two solutions were then combined. The combined solution was dried at 120 °C in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved precursor was carburized at a ramp rate of 15 °C/min to 900 °C for 2 hr under 30 sccm of H2 gas. After carburization the sample was cooled under 30 sccm N2 and passivated under 30 sccm 1% O2/N2 for 2 hours.",
     0.0, 0.0,             # Mo   wf, sigma
     76.6, 1.5,            # Mo2C wf, sigma
     23.4, 1.5,            # MoC  wf, sigma
     0.0, 0.0,             # MoO2 wf, sigma  (none reported)
     10.7, 0.932, 0.869,   # Rwp, GOF, chi2 (chi2 est. as GOF**2, not directly reported)
     "MoC.66"),
]

seed_procedures = {row[0] for row in seed_prompt_labels}

# Load dataset
data_path = "./dataset/Metal_Sucrose_DesignSpace_fr_T1_gpr_boicl.xlsx"
df = pd.read_excel(data_path, sheet_name="Design Space")

# ── Add new columns if they don't exist ───────────────────────────────────────
for col in ["MoC(Fm3m) weight fraction", "sigma.3", "MoC Ratio"]:
    if col not in df.columns:
        df[col] = None
        print(f"➕ Added column: {col}")

# ── Write seed results ─────────────────────────────────────────────────────────
for row in seed_prompt_labels:
    (procedure, mo_wf, mo_sig, mo2c_wf, mo2c_sig,
     moc_wf, moc_sig, moo2_wf, moo2_sig,
     rwp, gof, chi2, moc_ratio) = row
    match = df[df[PROMPT_COL] == procedure]
    if not match.empty:
        idx = match.index[0]
        df.at[idx, 'MoC(Fm3m) weight fraction']     = mo_wf
        df.at[idx, 'sigma']                           = mo_sig
        df.at[idx, 'Mo2C(pbcn) weight fraction']      = mo2c_wf
        df.at[idx, 'sigma.1']                         = mo2c_sig
        df.at[idx, 'MoC(Fm3m) weight fraction']       = moc_wf
        df.at[idx, 'sigma.2']                         = moc_sig
        df.at[idx, 'MoO2(P21/c) weight fraction']     = moo2_wf
        df.at[idx, 'sigma.3']                         = moo2_sig
        df.at[idx, 'Rwp']                             = rwp
        df.at[idx, 'GOF']                             = gof
        df.at[idx, 'X^2']                             = chi2
        df.at[idx, 'MoC Ratio']                       = moc_ratio
        print(f"✅ Updated index {idx} — MoC wf={moc_wf}, MoO2 wf={moo2_wf}, Rwp={rwp}")
    else:
        print(f"❌ Prompt not found: {procedure[:60]}...")

# Save back to xlsx
with pd.ExcelWriter(data_path, engine="openpyxl", mode="a",
                    if_sheet_exists="replace") as writer:
    df.to_excel(writer, sheet_name="Design Space", index=False)

# ── Shuffle and restrict to THIS trajectory's seeds only ──────────────────────
# NOTE: the shared xlsx also contains Traj 1 (EI)'s completed results
# (M16/17/21/23/24), and it also has duplicate rows for these exact 3
# procedures (e.g. M7 appears at both idx 5592 and 7776 — only one of the
# pair actually got written above). Requiring BOTH isin(seed_procedures) AND
# notna() ensures we pick up exactly the 3 written seed rows: not the EI
# trajectory's rows, and not the still-empty duplicate rows.
df_shuffled = df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

nonzero_mask    = df_shuffled[PROMPT_COL].isin(seed_procedures) & \
                  df_shuffled[COMPLETION_COL].notna()
nonzero_indices = df_shuffled.index[nonzero_mask].to_numpy()

print("Non-zero indices:", nonzero_indices[:])
print(f"Total non-zero entries: {len(nonzero_indices)}")
print(f"\n⏱️ Elapsed time: {time.time() - start_time:.2f} seconds")

✅ Updated index 5592 — MoC wf=72.1, MoO2 wf=0.0, Rwp=4.83
✅ Updated index 1239 — MoC wf=83.8, MoO2 wf=10.8, Rwp=4.16
✅ Updated index 7719 — MoC wf=23.4, MoO2 wf=0.0, Rwp=10.7
Non-zero indices: [4158 5225 6588]
Total non-zero entries: 3

⏱️ Elapsed time: 3.86 seconds


In [4]:
for i in nonzero_indices:
    print(df_shuffled[COMPLETION_COL][i])

23.4
72.1
83.8


In [5]:
# Ensure MoC weight fraction is numeric
df_shuffled[COMPLETION_COL] = pd.to_numeric(df_shuffled[COMPLETION_COL], errors="coerce")

# Row with the highest MoC cubic weight fraction (best observed so far)
index_max = df_shuffled[COMPLETION_COL].idxmax()
print(f"Row index with max MoC wf: {index_max}")
print(f"Max MoC wf: {df_shuffled.at[index_max, COMPLETION_COL]:.3f}")
print(f"Procedure: {df_shuffled.at[index_max, PROMPT_COL]}")
print(f"DataFrame length: {len(df_shuffled)}")

Row index with max MoC wf: 6588
Max MoC wf: 83.800
Procedure: 1 g of ammonium heptamolybdate tetrahydrate dissolved in 5.5 mL of DI water. Separately, 1 g of sucrose dissolved in 2 mL of DI water at 50 °C. The two solutions were then combined. The combined solution was dried at 120 °C in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved precursor was carburized at a ramp rate of 15 °C/min to 600 °C for 2 hr under 30 sccm of H2 gas. After carburization the sample was cooled under 30 sccm N2 and passivated under 30 sccm 1% O2/N2 for 2 hours.
DataFrame length: 7779


## Remove Ran Experiments

AskTellGPR needs its Isomap embedding fit *before* any points are told (it needs
more neighbors than we have labeled points otherwise). So the pool has to be
built here, ahead of the `asktell` construction below — not after it, as in
the EI notebook.

In [6]:
raw_data = df_shuffled.drop(nonzero_indices)
print("New dataset size:", len(raw_data))

New dataset size: 7776


In [7]:
import importlib, boicl.pool
importlib.reload(boicl.pool)
from boicl.pool import Pool
import cloudpickle, os, numpy as np, faiss, pickle

# GPR-specific cache filenames — distinct from the EI notebook's
# mo2c_sucrose_*.{pkl,npy,faiss} so the two trajectories never overwrite
# each other's pool cache (their raw_data sizes differ: this trajectory
# excludes only 3 seeds, EI excludes 8).
pool_path    = "mo2c_sucrose_gpr.pkl"
emb_path     = "mo2c_sucrose_gpr_emb.npy"
index_path   = "mo2c_sucrose_gpr_index.faiss"
prompts_path = "mo2c_sucrose_gpr_prompts.pkl"

prompts = raw_data['Procedure'].tolist()

if os.path.exists(prompts_path) and os.path.exists(emb_path) and os.path.exists(index_path):
    pool = Pool.from_prebuilt(
        prompts_path=prompts_path,
        emb_path=emb_path,
        index_path=index_path,
        formatter=lambda x: f"experimental procedure: {x}",
    )
    pool.reset()
    if len(pool._pool) != len(prompts):
        print(f"⚠️  Pool size mismatch — rebuilding ({len(prompts):,} procedures)...")
        pool = Pool(prompts, formatter=lambda x: f"experimental procedure: {x}")
        np.save(emb_path, pool._emb_matrix)
        faiss.write_index(pool.index, index_path)
        with open(prompts_path, "wb") as f:
            pickle.dump(pool._pool, f)
        print(f"✅ Pool rebuilt and saved ({len(prompts):,} procedures)")
    else:
        print(f"✅ Loaded pool from prebuilt files")
else:
    pool = Pool(prompts, formatter=lambda x: f"experimental procedure: {x}")
    np.save(emb_path, pool._emb_matrix)
    faiss.write_index(pool.index, index_path)
    with open(prompts_path, "wb") as f:
        pickle.dump(pool._pool, f)
    print(f"✅ Pool built and saved ({len(prompts):,} procedures)")

print(f"Pool size: {len(pool):,}")

✅ Loaded pool from prebuilt files
Pool size: 7,776


In [ ]:
system_message_path     = "./BOICL_docs/pred_system_message.txt"
inv_system_message_path = "./BOICL_docs/inv_system_message.txt"

assert os.path.exists(system_message_path), f"Missing: {system_message_path}"
assert os.path.exists(inv_system_message_path), f"Missing: {inv_system_message_path}"

with open(system_message_path, "r") as f:
    system_message = f.read()

with open(inv_system_message_path, "r") as f:
    inv_system_message = f.read()

print("✅ System messages loaded")
print(f"  pred: {len(system_message)} chars")
print(f"  inv:  {len(inv_system_message)} chars")
# NOTE: AskTellGPR.ask() always forces inv_filter=0, so it never calls
# inv_predict — inv_system_message is unused for GPR. Kept only so this
# cell and cell 15's ask() call stay drop-in compatible with the EI notebook.

# ── GPR surrogate (replaces the LLM few-shot model) ────────────────────────────
# x is embedded (text-embedding-ada-002) then projected via Isomap before
# fitting a SingleTaskGP — no LLM calls happen for prediction here.
# `pool` (built in the previous cell) is passed at construction so Isomap
# fits its manifold on the full ~7.7k-candidate design space rather than on
# just the 3 labeled seeds — with only 3 points and the default n_neighbors=5,
# telling would otherwise fail with "not enough data to train" (see
# boicl/asktellGPR.py's own error message for this exact situation).
GPR_CACHE_PATH = "./gpr_embeddings_cache.csv"

asktell = boicl.AskTellGPR(
    pool=pool,
    n_components=32,
    n_neighbors=5,
    cache_path=GPR_CACHE_PATH,
    prefix=(
        "The following are correctly answered questions. "
        "Each answer is numeric and ends with ###\n"
        "Note: the Mo:sucrose ratio is a variable parameter "
        "and should not be assumed constant across experiments."
    ),
    prompt_template=PromptTemplate(
        input_variables=["x", "y", "y_name"],
        template="Q: What is the {y_name} of {x}?@@@\nA: {y}###",
    ),
    suffix="What is the {y_name} of {x}?@@@\nA:",
    x_formatter=lambda x: (
    f"experimental procedure: {x['procedure']}. "
    f"Observed phases — "
    f"Mo weight fraction: {x['Mo_wf']:.1f}%, sigma: {x['Mo_sigma']:.3f}%. "
    f"Mo2C weight fraction: {x['Mo2C_wf']:.1f}%, sigma: {x['Mo2C_sigma']:.3f}%. "
    f"MoC weight fraction: {x['MoC_wf']:.1f}%, sigma: {x['MoC_sigma']:.3f}%. "
    f"MoO2 weight fraction: {x['MoO2_wf']:.1f}%, sigma: {x['MoO2_sigma']:.3f}%."
    ) if isinstance(x, dict) else x,
    y_name="cubic MoC weight fraction (%)",
    y_formatter=lambda y: f"{y:.1f}",
    x_name="synthesis procedure",
)

# ── Tell loop ───────────────────────────────────────────────────────────────
# Only refit the GP on the final tell() — otherwise it refits 3 times
# (once per point) for no benefit, since the manifold is already fixed by
# `pool` and every point will be told before we ever call ask().
for pos, i in enumerate(nonzero_indices):
    x = {
        'procedure':   df_shuffled['Procedure'].iloc[i],
        'Mo_wf':       float(df_shuffled['Mo (Im3m) weight fraction '].iloc[i] or 0.0),
        'Mo_sigma':    float(df_shuffled['sigma'].iloc[i] or 0.0),
        'Mo2C_wf':     float(df_shuffled['Mo2C(pbcn) weight fraction'].iloc[i] or 0.0),
        'Mo2C_sigma':  float(df_shuffled['sigma.1'].iloc[i] or 0.0),
        'MoC_wf':      float(df_shuffled['MoC(Fm3m) weight fraction'].iloc[i] or 0.0),
        'MoC_sigma':   float(df_shuffled['sigma.2'].iloc[i] or 0.0),
        'MoO2_wf':     float(df_shuffled['MoO2(P21/c) weight fraction'].iloc[i] or 0.0),
        'MoO2_sigma':  float(df_shuffled['sigma.3'].iloc[i] or 0.0),
    }
    y = float(df_shuffled['MoC(Fm3m) weight fraction'].iloc[i])
    is_last = pos == len(nonzero_indices) - 1
    asktell.tell(x, y, train=is_last)
    print("Told:", x['procedure'], y)

asktell.save_cache(GPR_CACHE_PATH)
print(f"✅ Embedding cache saved to {GPR_CACHE_PATH}")

✅ System messages loaded
  pred: 587 chars
  inv:  3552 chars
Cached embeddings not found. Creating new cache table.
Processing 7776 new items in 1556 batches...


In [ ]:
[new_prompt1], [aq_value], [mean], [std] = asktell.ask(
    pool,
    aq_fxn="expected_improvement",
    system_message=system_message,
    inv_system_message=inv_system_message,
)

# Remove from pool so it won't be suggested again
try:
    pool.choose(new_prompt1)
except ValueError:
    pass  # not in pool (inv_predict generated an unseen procedure)

# Look up by procedure string
match = df_shuffled[df_shuffled['Procedure'] == new_prompt1]

if len(match) > 0:
    print(match.iloc[0])
else:
    print("Procedure not found in df (may be a generated/unseen suggestion):")
    print(new_prompt1)

print(
    f"\033[1mAcquisition value:\033[0m {aq_value:.3f}\n"
    f"\033[1mMean prediction:\033[0m {mean:.3f}\n"
    f"\033[1mStd dev:\033[0m {std:.3f}"
)

Procedure                      1 g of ammonium heptamolybdate tetrahydrate di...
Carburize Temp (°C)                                                          690
Ramp Rate (°C/min)                                                            10
Purge Gas                                                                     N2
AMT:Sucrose Ratio                                                            0.5
Hold Time (hr)                                                               0.5
Flow Rate (sccm)                                                              30
Mo (Im3m) weight fraction                                                    0.0
sigma                                                                        0.0
Mo2C(pbcn) weight fraction                                                   0.0
sigma.1                                                                      0.0
MoC(Fm3m) weight fraction                                                   96.5
sigma.2                     

In [ ]:
new_prompt1

'1 g of ammonium heptamolybdate tetrahydrate dissolved in 4 mL of DI water. Separately, 2 g of sucrose dissolved in 3.5 mL of DI water at 50 °C. The two solutions were then combined. The combined solution was dried at 120 °C in air for 24 hours, then crushed and sieved to 212 microns. 0.6 g of the sieved precursor was carburized at a ramp rate of 10 °C/min to 690 °C for 0.5 hr under 30 sccm of N2 gas. After carburization the sample was cooled under 30 sccm N2 and passivated under 30 sccm 1% O2/N2 for 2 hours.'

## T1 current states

In [ ]:
import pandas as pd
import numpy as np
import re

# ── Parse synthesis params directly from procedure string ─────────────────────
def parse_params(proc):
    ramp  = re.search(r'ramp rate of ([\d.]+)', proc)
    temp  = re.search(r'to ([\d.]+) °C', proc)
    hold  = re.search(r'for ([\d.]+) hr', proc)
    gas   = re.search(r'under ([\d.]+ sccm of \w+) gas', proc)
    mo_g  = re.search(r'([\d.]+) g of ammonium heptamolybdate', proc)
    suc_g = re.search(r'([\d.]+) g of sucrose', proc)
    ratio_str = (
        f'{float(mo_g.group(1)):.1f} : {float(suc_g.group(1)):.1f}'
        if mo_g and suc_g else '—'
    )
    return {
        'Ramp\n(°C/min)':  float(ramp.group(1)) if ramp else None,
        'Hold Temp\n(°C)': float(temp.group(1)) if temp else None,
        'Hold Time\n(hr)': float(hold.group(1)) if hold else None,
        'Gas / Flow':      gas.group(1)          if gas  else None,
        'AMT:Suc\nRatio':  ratio_str,
    }

# ── Split seeds vs completed BO points ────────────────────────────────────────
# No BO-suggested results recorded yet for this trajectory — nonzero_indices
# is restricted (in cell 5) to only seed_procedures, so bo_rows is empty here.
bo_procedures = set()

seed_rows, bo_rows = [], []
for i in nonzero_indices:
    row  = df_shuffled.iloc[i]
    proc = row['Procedure']
    entry = {
        **parse_params(proc),
        'Cubic MoC\nwf (%)': float(row['MoC(Fm3m) weight fraction']),
        '_proc':              proc,
    }
    if proc in bo_procedures:
        bo_rows.append(entry)
    else:
        seed_rows.append(entry)

for k, d in enumerate(seed_rows):
    d['Label']      = ['i', 'ii', 'iii', 'iv', 'v'][k]
    d['Trajectory'] = 'Seed'

# reverse so latest run (added last to completed_prompt_labels) = label 1
bo_rows_labeled = list(reversed(bo_rows))
for k, d in enumerate(bo_rows_labeled):
    d['Label']      = str(k + 1)
    d['Trajectory'] = 'Traj GPR'

# ── Next suggested point (not yet measured) ───────────────────────────────────
next_point = {
    'Label':             'next →',
    'Trajectory':        'Traj GPR',
    **parse_params(new_prompt1),
    'Cubic MoC\nwf (%)': np.nan,
    '_proc':             new_prompt1,
}

# ── Assemble ──────────────────────────────────────────────────────────────────
all_rows    = seed_rows + bo_rows_labeled + [next_point]
full_df     = pd.DataFrame(all_rows).set_index('Label')
proc_series = full_df['_proc']

summary = full_df[[
    'Trajectory', 'Ramp\n(°C/min)', 'Hold Temp\n(°C)',
    'Hold Time\n(hr)', 'Gas / Flow', 'AMT:Suc\nRatio', 'Cubic MoC\nwf (%)'
]]

# ── Styling ───────────────────────────────────────────────────────────────────
def color_traj(val):
    return {
        'Seed':      'background-color: #d0e8ff; color: #1a3a5c',
        'Traj GPR':  'background-color: #d4edda; color: #155724',
    }.get(str(val), '')

def bar_moc(val):
    try:
        pct = float(val) / 100
        return (
            f'background: linear-gradient(90deg, #4e91d2 {pct*100:.0f}%, '
            f'transparent {pct*100:.0f}%); color: #000; font-weight: bold'
        )
    except (TypeError, ValueError):
        return 'color: #999; font-style: italic'

def highlight_next(row):
    if row.name == 'next →':
        return ['border-top: 2px dashed #e67e22; color: #e67e22; font-weight: bold'] * len(row)
    return [''] * len(row)

styled = (
    summary.style
    .map(color_traj, subset=['Trajectory'])
    .map(bar_moc,    subset=['Cubic MoC\nwf (%)'])
    .apply(highlight_next, axis=1)
    .format({
        'Cubic MoC\nwf (%)': lambda v: f'{float(v):.1f}' if pd.notna(v) else '⏳ pending',
        'AMT:Suc\nRatio':     lambda v: str(v)     if pd.notna(v) else '—',
        'Ramp\n(°C/min)':     lambda v: f'{v:.0f}' if pd.notna(v) else '—',
        'Hold Temp\n(°C)':    lambda v: f'{v:.0f}' if pd.notna(v) else '—',
        'Hold Time\n(hr)':    lambda v: f'{v:.1f}' if pd.notna(v) else '—',
        'Gas / Flow':         lambda v: str(v)      if pd.notna(v) else '—',
    })
    .set_caption('BOICL Campaign — GPR  |  Cubic MoC (Fm3̄m) Optimization')
    .set_table_styles([
        {'selector': 'caption',
         'props': [('font-size', '14px'), ('font-weight', 'bold'),
                   ('text-align', 'left'), ('padding', '0 0 8px 0')]},
        {'selector': 'th',
         'props': [('background-color', '#2c5f8a'), ('color', 'white'),
                   ('padding', '8px 12px'), ('text-align', 'center'),
                   ('white-space', 'pre-line'), ('line-height', '1.3')]},
        {'selector': 'td',
         'props': [('padding', '6px 12px'), ('text-align', 'center'),
                   ('border-bottom', '1px solid #e0e0e0')]},
        {'selector': 'tr:hover td',
         'props': [('filter', 'brightness(0.95)')]},
    ])
)

display(styled)

# ── Summary stats ─────────────────────────────────────────────────────────────
measured  = summary['Cubic MoC\nwf (%)'].apply(pd.to_numeric, errors='coerce').dropna()
best_val  = measured.max()
best_lbl  = measured.idxmax()
best_traj = summary.loc[best_lbl, 'Trajectory']

print(f"\n🏆 Best so far: {best_val:.1f}%  (point {best_lbl} — {best_traj})")

# ── Full procedure strings ────────────────────────────────────────────────────
print("\n📋 Procedure strings:")
for label, proc in proc_series.items():
    moc     = summary.loc[label, 'Cubic MoC\nwf (%)']
    moc_str = f"{float(moc):.1f}%" if pd.notna(moc) else "⏳ pending"
    traj    = summary.loc[label, 'Trajectory']
    print(f"\n  [{label}]  {traj}  —  {moc_str}")
    print(f"  {proc}")

In [1]:
import pandas as pd
data_path = "./dataset/Metal_Sucrose_DesignSpace_fr_T1.xlsx"
df = pd.read_excel(data_path)
for col in df.columns[1:7]:
    print(col, "->", df[col].unique())

df.columns



Carburize Temp (°C) -> [550 560 570 580 590 600 610 620 630 640 650 660 670 680 690 700 710 720
 730 740 750 760 770 780 790 800 810 820 830 840 850 860 870 880 890 900]
Ramp Rate (°C/min) -> [ 5 10 15]
Purge Gas -> ['H2' 'N2']
AMT:Sucrose Ratio -> [0.5 1.  2. ]
Hold Time (hr) -> [ 0.5  2.   5.  10. ]
Flow Rate (sccm) -> [ 30  60 100]


Index(['Procedure', 'Carburize Temp (°C)', 'Ramp Rate (°C/min)', 'Purge Gas',
       'AMT:Sucrose Ratio', 'Hold Time (hr)', 'Flow Rate (sccm)',
       'Mo (Im3m) weight fraction ', 'sigma', 'Mo2C(pbcn) weight fraction',
       'sigma.1', 'MoC(Fm3m) weight fraction', 'sigma.2', 'Rwp', 'GOF', 'X^2',
       'MoO2(P21/c) weight fraction', 'sigma.3', 'MoC Ratio'],
      dtype='object')